In [28]:
from tensorflow.keras.datasets import imdb
(train_input, train_target),(test_input,test_target)=imdb.load_data(num_words=500)

/opt/miniconda3/lib/python3.13/site-packages/numpy/lib/_format_impl.py:838: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  array = pickle.load(fp, **pickle_kwargs)


In [34]:
from sklearn.model_selection import train_test_split
train_input,val_input,train_target,val_target = train_test_split(
    train_input,train_target,test_size=0.2,random_state=42
)

In [36]:
import numpy as np
lengths=np.array([len(x) for x in train_input])
print(np.mean(lengths),np.median(lengths))#평균>중앙값

239.00925 178.0


In [38]:
from tensorflow.keras.preprocessing.sequence import pad_sequences

train_seq=pad_sequences(train_input,maxlen=100)
val_seq=pad_sequences(val_input,maxlen=100)

In [43]:
from tensorflow import keras
model=keras.Sequential()
model.add(keras.layers.Embedding(500,16))
model.add(keras.layers.LSTM(8))
model.add(keras.layers.Dense(1,activation='sigmoid'))

model.build(input_shape=(None,100))
model.summary()

Model: "sequential_4"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_4 (Embedding)         │ (None, 100, 16)        │         8,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_3 (LSTM)                   │ (None, 8)              │           800 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 1)              │             9 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 8,809 (34.41 KB)

 Trainable params: 8,809 (34.41 KB)

 Non-trainable params: 0 (0.00 B)

In [44]:
#테스트 세트
test_seq = pad_sequences(test_input, maxlen=100)
print(train_seq.shape,val_seq.shape,test_seq.shape)#원핫인코딩과 전혀 상관없음

(20000, 100) (5000, 100) (25000, 100)


In [45]:
model.summary()#500*16??=8000

Model: "sequential_4"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_4 (Embedding)         │ (None, 100, 16)        │         8,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_3 (LSTM)                   │ (None, 8)              │           800 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 1)              │             9 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 8,809 (34.41 KB)

 Trainable params: 8,809 (34.41 KB)

 Non-trainable params: 0 (0.00 B)

#LSTM 신경망 훈련하기

In [46]:
#compile
rmsprop=keras.optimizers.RMSprop(learning_rate=1e-4)#0.0001
model.compile(
    optimizer=rmsprop,#adam도 가능
    loss='binary_crossentropy',
    metrics=['accuracy']
)
checkpoint_cb=keras.callbacks.ModelCheckpoint("../Data/best-lstm-model.keras")#저장 내가 안해 너가 돌려보고 좋은거 저장해줘
early_stopping_cb=keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True)
history=model.fit(
    train_seq,
    train_target,
    epochs=100,
    batch_size=32,
    validation_data=(val_seq, val_target),
    callbacks=[checkpoint_cb,early_stopping_cb]
)

Epoch 1/100
625/625 ━━━━━━━━━━━━━━━━━━━━ 6s 9ms/step - accuracy: 0.5430 - loss: 0.6921 - val_accuracy: 0.6036 - val_loss: 0.6899
Epoch 2/100
625/625 ━━━━━━━━━━━━━━━━━━━━ 5s 9ms/step - accuracy: 0.6526 - loss: 0.6779 - val_accuracy: 0.6968 - val_loss: 0.6446
Epoch 3/100
625/625 ━━━━━━━━━━━━━━━━━━━━ 5s 9ms/step - accuracy: 0.7102 - loss: 0.5933 - val_accuracy: 0.7176 - val_loss: 0.5696
Epoch 4/100
625/625 ━━━━━━━━━━━━━━━━━━━━ 5s 8ms/step - accuracy: 0.7412 - loss: 0.5484 - val_accuracy: 0.7502 - val_loss: 0.5350
Epoch 5/100
625/625 ━━━━━━━━━━━━━━━━━━━━ 6s 9ms/step - accuracy: 0.7635 - loss: 0.5171 - val_accuracy: 0.7664 - val_loss: 0.5096
Epoch 6/100
625/625 ━━━━━━━━━━━━━━━━━━━━ 6s 9ms/step - accuracy: 0.7788 - loss: 0.4927 - val_accuracy: 0.7800 - val_loss: 0.4893
Epoch 7/100
625/625 ━━━━━━━━━━━━━━━━━━━━ 5s 9ms/step - accuracy: 0.7880 - loss: 0.4738 - val_accuracy: 0.7868 - val_loss: 0.4724
Epoch 8/100
625/625 ━━━━━━━━━━━━━━━━━━━━ 5s 9ms/step - accuracy: 0.7940 - loss: 0.4593 - val_accu

In [48]:
val_loss,val_acc= model.evaluate(val_seq,val_target)
test_loss,test_acc=model.evaluate(test_seq, test_target)

print(f"검증세트-loss:{val_loss:.4f}, accuracy:{val_acc:.4f}")
print(f"테스트세트-loss:{test_loss:.4f}, accuracy:{test_acc:.4f}")

157/157 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7998 - loss: 0.4292
782/782 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.8048 - loss: 0.4241
검증세트-loss:0.4292, accuracy:0.7998
테스트세트-loss:0.4241, accuracy:0.8048


In [49]:
from tensorflow import keras
model=keras.Sequential()
model.add(keras.layers.Embedding(500,16, input_length=100))
model.add(keras.layers.LSTM(8, dropout=0.3))
model.add(keras.layers.Dense(1,activation='sigmoid'))

model.summary()

/opt/miniconda3/lib/python3.13/site-packages/keras/src/layers/core/embedding.py:103: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential_5"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_5 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_4 (LSTM)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [50]:
#compile
rmsprop=keras.optimizers.RMSprop(learning_rate=1e-4)#0.0001
model.compile(
    optimizer=rmsprop,#adam도 가능
    loss='binary_crossentropy',
    metrics=['accuracy']
)
checkpoint_cb=keras.callbacks.ModelCheckpoint("../Data/best-lstm-model.keras")#저장 내가 안해 너가 돌려보고 좋은거 저장해줘
early_stopping_cb=keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True)
history=model.fit(
    train_seq,
    train_target,
    epochs=100,
    batch_size=32,
    validation_data=(val_seq, val_target),
    callbacks=[checkpoint_cb,early_stopping_cb]
)

Epoch 1/100
625/625 ━━━━━━━━━━━━━━━━━━━━ 8s 10ms/step - accuracy: 0.5400 - loss: 0.6922 - val_accuracy: 0.6086 - val_loss: 0.6907
Epoch 2/100
625/625 ━━━━━━━━━━━━━━━━━━━━ 6s 10ms/step - accuracy: 0.6047 - loss: 0.6866 - val_accuracy: 0.6726 - val_loss: 0.6775
Epoch 3/100
625/625 ━━━━━━━━━━━━━━━━━━━━ 6s 9ms/step - accuracy: 0.6819 - loss: 0.6301 - val_accuracy: 0.7164 - val_loss: 0.5821
Epoch 4/100
625/625 ━━━━━━━━━━━━━━━━━━━━ 6s 9ms/step - accuracy: 0.7171 - loss: 0.5682 - val_accuracy: 0.7430 - val_loss: 0.5450
Epoch 5/100
625/625 ━━━━━━━━━━━━━━━━━━━━ 6s 9ms/step - accuracy: 0.7485 - loss: 0.5311 - val_accuracy: 0.7576 - val_loss: 0.5129
Epoch 6/100
625/625 ━━━━━━━━━━━━━━━━━━━━ 6s 10ms/step - accuracy: 0.7654 - loss: 0.5027 - val_accuracy: 0.7706 - val_loss: 0.4925
Epoch 7/100
625/625 ━━━━━━━━━━━━━━━━━━━━ 6s 10ms/step - accuracy: 0.7794 - loss: 0.4793 - val_accuracy: 0.7852 - val_loss: 0.4708
Epoch 8/100
625/625 ━━━━━━━━━━━━━━━━━━━━ 6s 10ms/step - accuracy: 0.7894 - loss: 0.4647 - val

In [ ]:
val_loss,val_acc= model.evaluate(val_seq,val_target)
test_loss,test_acc=model.evaluate(test_seq, test_target)

print(f"검증세트-loss:{val_loss:.4f}, accuracy:{val_acc:.4f}")
print(f"테스트세트-loss:{test_loss:.4f}, accuracy:{test_acc:.4f}")
#모델 앞에 데이터 만드는 것이 제일 중요 500개의 데이터 